# Task 3: Comparison of Agreement Metrics
### Percentage Agreement, Bennett's S, Cohen's $\kappa$, and Gwet's $AC_1$

In this notebook, we compare 4 popular inter-annotator agreement metrics:
1. **Percentage Agreement ($P_o$)**: Simple proportion of concordant items.
2. **Bennett, Alpert, and Goldstein's $S$**: Chance-correction assuming uniform category distribution ($P_e = 1/k = 0.5$).
3. **Cohen's $\kappa$ (Kappa)**: Chance-correction using observed empirical marginal distributions.
4. **Gwet's $AC_1$**: Chance-correction designed to be robust against skewed marginal prevalence.

We evaluate all 4 metrics:
- **Part 1**: On the real **example dataset** (Batch 14, Batch 43, All Data, Before & After discussion).
- **Part 2**: On **fake data** across different prevalences ($N = 100$).


In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120


## 1. Metric Implementations

We implement the four agreement formulas for binary classification (`"offensive"` vs `"not offensive"`):


In [ ]:
def calc_percentage_agreement(a1, a2):
    """Raw observed agreement."""
    return np.mean(a1 == a2)

def calc_bennett_s(a1, a2, k=2):
    """Bennett, Alpert, and Goldstein (1954) S coefficient."""
    po = calc_percentage_agreement(a1, a2)
    return 1.0 - (k * (1.0 - po)) / (k - 1.0)

def calc_cohen_kappa(a1, a2):
    """Cohen's Kappa (1960)."""
    po = calc_percentage_agreement(a1, a2)
    
    # marginal probabilities of 'offensive'
    p1_off = np.mean(a1 == "offensive")
    p2_off = np.mean(a2 == "offensive")
    
    # expected chance agreement based on marginals
    pe = (p1_off * p2_off) + ((1.0 - p1_off) * (1.0 - p2_off))
    
    if pe == 1.0:
        return 1.0
    return (po - pe) / (1.0 - pe)

def calc_gwet_ac1(a1, a2):
    """Gwet's AC1 (2002/2008)."""
    po = calc_percentage_agreement(a1, a2)
    
    # average probability of 'offensive'
    pi_off = (np.mean(a1 == "offensive") + np.mean(a2 == "offensive")) / 2.0
    
    # Gwet's Pe for k=2 categories
    pe = 2.0 * pi_off * (1.0 - pi_off)
    
    if pe == 1.0:
        return 1.0
    return (po - pe) / (1.0 - pe)

def compute_all_metrics(a1, a2):
    return {
        "PA (Po)": calc_percentage_agreement(a1, a2),
        "Bennett's S": calc_bennett_s(a1, a2),
        "Cohen's Kappa": calc_cohen_kappa(a1, a2),
        "Gwet's AC1": calc_gwet_ac1(a1, a2)
    }


## 2. Evaluation on Example Data

We load `batch_14.jsonl` and `batch_43.jsonl` and compute all 4 metrics before and after discussion.


In [ ]:
# find example data directory
candidates = [
    "/scratch/group_01/bsem-group/annotation_experiment/example_data/",
    "./example_data/",
    "../annotation_experiment/example_data/",
    "example_data/"
]
data_dir = next((p for p in candidates if os.path.exists(p)), None)
print(f"Loading data from: {data_dir}")

records = []
for f in sorted(glob.glob(os.path.join(data_dir, "*.jsonl"))):
    with open(f, "r", encoding="utf-8") as fp:
        for line in fp:
            line = line.strip()
            if line:
                records.append(json.loads(line))

df_ex = pd.DataFrame(records)

# helper to extract labels
def get_labels_pair(df_subset, stage="before"):
    a1, a2 = [], []
    for _, row in df_subset.iterrows():
        if stage == "before":
            l1 = "offensive" if "offensive" in row["labels_annotation"][0]["labels"] else "not offensive"
            l2 = "offensive" if "offensive" in row["labels_annotation"][1]["labels"] else "not offensive"
        else:
            disc_dict = {d["user"]: d["labels"] for d in row.get("labels_discussion", [])}
            u1 = row["labels_annotation"][0]["user"]
            u2 = row["labels_annotation"][1]["user"]
            l1 = "offensive" if "offensive" in disc_dict.get(u1, row["labels_annotation"][0]["labels"]) else "not offensive"
            l2 = "offensive" if "offensive" in disc_dict.get(u2, row["labels_annotation"][1]["labels"]) else "not offensive"
        a1.append(l1)
        a2.append(l2)
    return np.array(a1), np.array(a2)


In [ ]:
subsets = {
    "All Data (81)": df_ex,
    "Batch 14 (30)": df_ex[df_ex["batch_number"] == 14],
    "Batch 43 (51)": df_ex[df_ex["batch_number"] == 43]
}

ex_rows = []
for name, sub in subsets.items():
    for stage in ["before", "after"]:
        a1, a2 = get_labels_pair(sub, stage)
        m = compute_all_metrics(a1, a2)
        ex_rows.append({
            "Subset": name,
            "Stage": stage.capitalize(),
            "PA (Po)": f"{m['PA (Po)']:.4f}",
            "Bennett's S": f"{m["Bennett's S"]:.4f}",
            "Cohen's Kappa": f"{m["Cohen's Kappa"]:.4f}",
            "Gwet's AC1": f"{m["Gwet's AC1"]:.4f}"
        })

df_ex_results = pd.DataFrame(ex_rows)
print("=== Agreement Metrics on Example Data ===")
df_ex_results


## 3. Evaluation on Fake Data Across Prevalences ($N = 100$)

We generate synthetic data for different prevalences of `"offensive"` from $0.0$ to $1.0$ (step size 0.1).
We compare how the 4 metrics respond to the changing prevalence.


In [ ]:
np.random.seed(42)
n_samples = 100
prevalences = np.linspace(0.0, 1.0, 11)

sim_metrics = []

for p in prevalences:
    # random labels for two independent annotators with prevalence p
    ann1 = np.random.choice(["offensive", "not offensive"], size=n_samples, p=[p, 1.0 - p])
    ann2 = np.random.choice(["offensive", "not offensive"], size=n_samples, p=[p, 1.0 - p])
    
    m = compute_all_metrics(ann1, ann2)
    sim_metrics.append({
        "prevalence": p,
        "Percentage Agreement": m["PA (Po)"],
        "Bennett's S": m["Bennett's S"],
        "Cohen's Kappa": m["Cohen's Kappa"],
        "Gwet's AC1": m["Gwet's AC1"]
    })

df_sim_metrics = pd.DataFrame(sim_metrics)
df_sim_metrics


## 4. Multi-Metric Comparison Plot


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df_sim_metrics["prevalence"], df_sim_metrics["Percentage Agreement"], marker="s", linestyle="--", color="#3498db", label="Percentage Agreement ($P_o$)", linewidth=2)
plt.plot(df_sim_metrics["prevalence"], df_sim_metrics["Bennett's S"], marker="o", color="#e67e22", label="Bennett's S", linewidth=2.5)
plt.plot(df_sim_metrics["prevalence"], df_sim_metrics["Cohen's Kappa"], marker="^", color="#e74c3c", label="Cohen's $\kappa$", linewidth=2)
plt.plot(df_sim_metrics["prevalence"], df_sim_metrics["Gwet's AC1"], marker="d", color="#2ecc71", label="Gwet's $AC_1$", linewidth=2)

plt.axhline(0, color="gray", linestyle=":", alpha=0.7)
plt.xlabel("Prevalence of 'offensive' label", fontsize=11)
plt.ylabel("Metric Score", fontsize=11)
plt.title("Task 3: Comparison of Agreement Metrics Across Prevalences (Fake Data, N=100)", fontsize=13, fontweight="bold")
plt.xticks(prevalences)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=10, loc="lower center")

plt.tight_layout()
plt.show()


## 5. Discussion & Theoretical Comparison

### 1. Findings on Real Example Data:
* **Ranking of scores**:
  On All Data (Before Discussion):
  - **Percentage Agreement**: $0.9136$ (highest because it ignores chance agreement)
  - **Gwet's $AC_1$**: $0.8748$ (high and balanced)
  - **Bennett's $S$**: $0.8272$ (penalized by fixed $P_e = 0.5$)
  - **Cohen's $\kappa$**: $0.7214$ (lowest because the severe label imbalance $80\% / 20\%$ inflates $P_e$)
* Discussion raised all metrics substantially: After discussion, $P_o$ reached $0.9753$, Bennett's $S$ reached $0.9506$, and Cohen's $\kappa$ reached $0.9221$.

### 2. Findings on Fake Data Across Prevalences:
* **Cohen's $\kappa$**:
  Since both annotators are just guessing independently with probability $p$, their observed agreement is entirely due to chance ($P_o \approx P_e$). Thus, **Cohen's $\kappa$ stays near $0.0$** across all prevalences. It correctly identifies that the annotators have zero true agreement above chance!
* **Percentage Agreement ($P_o$)**:
  Forms a U-shaped curve that reaches $50\%$ at $p = 0.5$ and shoots up to $100\%$ at $p=0$ and $p=1$. This illustrates why raw percentage agreement is misleading when classes are imbalanced.
* **Bennett's $S$**:
  Because Bennett's formula assumes a fixed uniform expected agreement ($P_e = 0.5$), Bennett's $S$ mirrors the U-shaped curve of $P_o$ ($S = 2P_o - 1$). At the extremes, it suggests positive agreement even though annotators are only rolling dice!
* **Gwet's $AC_1$**:
  Gwet's $AC_1$ adapts expected chance agreement $P_e = 2\pi(1-\pi)$, balancing between sensitivity to guessing and resistance to the prevalence paradox.
